# 08 · Notebook 07, node by node

Runs each node of notebook 07 on its own, in the graph's order, on one shared `state`.
- Run the cells from top to bottom. Re-run a cell to repeat its node.
- `run(node)` merges the node's update into `state` the same way the graph does.
- After a failure, `run` skips the node. Start again from preflight (1).
- The microscope's FOV is never changed.

In [ ]:
%matplotlib inline
from pathlib import Path

from IPython.display import Image, display

from stem_agent.config import auto_focus_settings, sam_settings
from stem_agent.nodes.auto_focus_node import auto_focus_node
from stem_agent.nodes.eds_node import eds_node, eds_settings
from stem_agent.nodes.overview_node import overview_node
from stem_agent.nodes.preflight_node import preflight_node
from stem_agent.nodes.sam_segment_node import sam_segment_node
from stem_agent.nodes.save_node import save_node
from stem_agent.nodes.status_node import status_node

SETTINGS = {
    "instrument": "real",
    "run_tag": "nodes_07",
    "auto_focus": {},                 # {} = defaults; no fov_m = keep the microscope's FOV
    "sam": {"min_visible": 0.5, "min_diameter_nm": 2.0, "max_diameter_nm": 30.0},
    "max_particles": 10,              # EDS on the 10 largest particles
}
assert "auto_focus" in SETTINGS, 'without "auto_focus" the node is skipped'
auto_focus_settings(SETTINGS), sam_settings(SETTINGS), eds_settings(SETTINGS)   # a typo stops here


async def run(node, always=False) -> bool:
    """Run one node on `state` and merge its update as the graph does. -> True if it worked."""
    if state.get("run_status") == "failed" and not always:
        print("SKIPPED: the run failed. Start again from preflight (1).")
        return False
    update = await node(state)
    errors = state.get("errors", []) + update.get("errors", [])            # errors add up
    tuning = {**state.get("tuning", {}), **update.get("tuning", {})}       # tuning merges
    state.update(update, errors=errors, tuning=tuning)
    print("->", sorted(update), "| run_status:", state.get("run_status"))
    for error in update.get("errors", []):
        print("ERROR:", error)
    return "errors" not in update


def show(*paths):
    for path in paths:
        if Path(path).exists():
            display(Image(str(path), width=420))

## 1 · Preflight: MCP, Tiled, new run folder

In [ ]:
state = {"settings": SETTINGS}            # a new, empty run
if await run(preflight_node):
    print(state["run"]["run_dir"])

## 2 · Status: the readiness gate (a microscope that is not ready fails the run)

In [ ]:
if await run(status_node):
    snap = state["snapshots"][-1]
    print({key: snap[key] for key in ("ready", "fov_m", "defocus_m", "problems", "warnings")})

## 3 · Image before: `images[0]`

In [ ]:
if await run(overview_node):
    image = state["overview"]
    print(f"image {image['index']}: FOV {image['fov_m'] * 1e9:.1f} nm, contrast {image['metrics']['contrast']:.4g}")
    show(image["preview_path"])

## 4 · Auto focus: OptiSTEM C1/A1
**Changes focus and stigmation.** If the image gets worse, the node puts the defocus back. A1 stays as OptiSTEM left it.

In [ ]:
if await run(auto_focus_node):
    focus = state["tuning"]["auto_focus"]
    print(f"{focus['stop_reason']}: defocus {focus['best_offset_m'][0] * 1e9:+.1f} nm, "
          f"{focus['settings']['metric']} {focus['best_score']:.4g}")
    show(Path(state["run"]["run_dir"]) / "auto_focus_before.png", focus["figure_path"])

## 5 · Image after: `images[1]`, the image that segmentation and EDS use

In [ ]:
if await run(overview_node):
    before, after = state["images"][0], state["images"][-1]
    print(f"contrast {before['metrics']['contrast']:.4g} -> {after['metrics']['contrast']:.4g}")
    print(f"FOV {before['fov_m'] * 1e9:.1f} -> {after['fov_m'] * 1e9:.1f} nm (should not change)")
    show(after["preview_path"])

## 6 · SAM segmentation of the latest image
To tune it, change `SETTINGS["sam"]` and re-run this cell.

In [ ]:
# SETTINGS["sam"]["max_diameter_nm"] = 20.0
if await run(sam_segment_node):
    print(len(state["particles"]), "particles, area px:", [p.get("area_px") for p in state["particles"]])
    show(state["segmentation"]["overlay_path"])

## 7 · EDS on the particles, largest first, up to `max_particles`
**Moves the beam onto each particle.**

In [ ]:
if await run(eds_node):
    print(len(state["spectra"]), "spectra")
    show(Path(state["run"]["run_dir"]) / "eds.png")

## 8 · Final status and save: `summary.json`, `summary.png`
These run even after a failure, as they do in the graph.

In [ ]:
await run(status_node, always=True)
if await run(save_node, always=True):
    print(state["run_status"], "|", state["summary_path"])
    show(Path(state["run"]["run_dir"]) / "summary.png")